In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def backproject_depth_torch(depth, inv_K):
    B, _, H, W = depth.shape
    device = depth.device

    u = torch.arange(0, W, device=device)
    v = torch.arange(0, H, device=device)
    grid_u, grid_v = torch.meshgrid(u, v, indexing='xy')

    ones = torch.ones_like(grid_u)
    pix_coords = torch.stack([grid_u, grid_v, ones], dim=0).float()  # (3, H, W)
    pix_coords = pix_coords.reshape(3, -1).unsqueeze(0).repeat(B, 1, 1)  # (B, 3, H*W)

    cam_dirs = torch.bmm(inv_K, pix_coords)  # (B, 3, H*W)
    depth_flat = depth.view(B, 1, -1)

    cam_points = cam_dirs * depth_flat  # (B, 3, H*W)
    cam_points = cam_points.view(B, 3, H, W)
    return cam_points

def get_stereo_M_t2s_torch(baseline=0.12, device='cpu'):
    M = torch.eye(4, device=device).unsqueeze(0)
    M[:, 0, 3] = -baseline
    return M

def reproject_to_source_torch(cam_points, K, M_t2s):
    B, _, H, W = cam_points.shape
    device = cam_points.device

    cam_points_flat = cam_points.view(B, 3, -1)  # (B, 3, H*W)
    ones = torch.ones((B, 1, H * W), device=device)
    cam_points_homo = torch.cat([cam_points_flat, ones], dim=1)  # (B, 4, H*W)

    cam_points_src = torch.bmm(M_t2s, cam_points_homo)[:, :3, :]  # (B, 3, H*W)

    pixels = torch.bmm(K, cam_points_src)  # (B, 3, H*W)
    p0 = pixels[:, :2, :] / (pixels[:, 2:3, :] + 1e-8)  # (B, 2, H*W)
    p0 = p0.view(B, 2, H, W).permute(0, 2, 3, 1)  # (B, H, W, 2)
    return p0

def warp_depth_to_source_view(depth, K, inv_K, M_t2s):
    cam_points = backproject_depth_torch(depth, inv_K)
    p0 = reproject_to_source_torch(cam_points, K, M_t2s)

    B, _, H, W = depth.shape
    p0_norm = p0.clone()
    p0_norm[..., 0] = (p0[..., 0] / (W - 1)) * 2 - 1
    p0_norm[..., 1] = (p0[..., 1] / (H - 1)) * 2 - 1

    depth_warped = F.grid_sample(depth, p0_norm, mode='bilinear', align_corners=True)
    return depth_warped, cam_points, p0

def check_consistency(depth_tensor, K, baseline=0.12, z_thresh=1.0, visualize=True):
    """
    Check 3D consistency of a depth map by warping it through stereo transformation.
    
    Args:
        depth_np (np.ndarray): Depth map (H, W) in numpy array
        K (np.ndarray): Intrinsic matrix (3, 3)
        baseline (float): Stereo baseline in meters
        z_thresh (float): Threshold for z-difference to consider consistent
        visualize (bool): If True, displays the consistency mask
        
    Returns:
        consistency_mask (torch.Tensor): (H, W) mask where values < z_thresh
    """
    assert depth_np.ndim == 2, "Depth map must be a 2D array"
    
    H, W = depth_np.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inv_K_tensor = torch.inverse(K)

    # 1. Get stereo transformation matrix
    M_t2s = get_stereo_M_t2s_torch(baseline=baseline, device=device)
    M_s2t = torch.inverse(M_t2s)

    # 2. Warp depth from target to source view
    depth_warped_to_src, cam_points, p0 = warp_depth_to_source_view(depth_tensor, K_tensor, inv_K_tensor, M_t2s)

    # 3. Backproject warped depth to source 3D
    cam_points_src = backproject_depth_torch(depth_warped_to_src, inv_K_tensor)

    # 4. Transform back to target coordinate system
    cam_points_src_flat = cam_points_src.view(1, 3, -1)
    ones = torch.ones((1, 1, H * W), device=device)
    cam_points_src_homo = torch.cat([cam_points_src_flat, ones], dim=1)
    cam_points_src_to_target = torch.bmm(M_s2t, cam_points_src_homo)[:, :3, :].view(1, 3, H, W)

    # 5. Z-difference
    z_diff = torch.abs(cam_points[0, 2] - cam_points_src_to_target[0, 2])
    consistency_mask = (z_diff < z_thresh).float()

    # Optional visualization
    if visualize:
        plt.imshow(consistency_mask.cpu(), cmap='gray')
        plt.title("3D Consistency Mask")
        plt.axis("off")
        plt.show()

    return consistency_mask


In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from glob import glob
# from tqdm import tqdm

# def process_consistency_folder(
#     depth_folder, 
#     K, 
#     baseline=0.12, 
#     z_thresh=1.0, 
#     save_folder=None, 
#     visualize=False
# ):
#     """
#     Applies 3D consistency check to all .npy depth maps in a folder.

#     Args:
#         depth_folder (str): Path to folder containing depth .npy files
#         K (np.ndarray): Camera intrinsics (3x3)
#         baseline (float): Stereo baseline
#         z_thresh (float): Z-consistency threshold
#         save_folder (str or None): If given, saves masked depth maps here
#         visualize (bool): If True, displays each masked depth
#     """
#     depth_files = sorted(glob(os.path.join(depth_folder, "*.npy")))

#     if save_folder:
#         os.makedirs(save_folder, exist_ok=True)

#     for depth_path in tqdm(depth_files, desc="Processing depth maps"):
#         filename = os.path.basename(depth_path)
#         depth_np = np.load(depth_path)  # (H, W)

#         mask = check_consistency(depth_np, K, baseline=baseline, z_thresh=z_thresh, visualize=False)
#         mask_np = mask.cpu().numpy().astype(bool)

#         # Masked depth
#         consistency_gt = depth_np.copy()
#         consistency_gt[~mask_np] = 0
#         # print(np.unique(depth_np))
#         # print(np.unique(consistency_gt))
#         # print()
#         if save_folder:
#             save_path = os.path.join(save_folder, filename)
#             np.save(save_path, consistency_gt)

#         if visualize:
#             plt.figure(figsize=(8, 4))
#             plt.imshow(consistency_gt, cmap="magma_r")
#             plt.title(f"Masked depth: {filename}")
#             plt.axis("off")
#             plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm

def process_consistency(
    depth_files, 
    K, 
    baseline=0.12, 
    z_thresh=1.0, 
    save_folder=None, 
    visualize=False
):
    """
    Applies 3D consistency check to all .npy depth maps in a folder.

    Args:
        depth_folder (str): Path to folder containing depth .npy files
        K (np.ndarray): Camera intrinsics (3x3)
        baseline (float): Stereo baseline
        z_thresh (float): Z-consistency threshold
        save_folder (str or None): If given, saves masked depth maps here
        visualize (bool): If True, displays each masked depth
    """

    if save_folder:
        os.makedirs(save_folder, exist_ok=True)

    mask = check_consistency(depth_files, K, baseline=baseline, z_thresh=z_thresh, visualize=False)
    mask_np = mask.cpu().numpy().astype(bool)

    # Masked depth
    consistency_gt = depth_np.copy()
    consistency_gt[~mask_np] = 0
    # print(np.unique(depth_np))
    # print(np.unique(consistency_gt))
    # print()
    return consistency_gt


In [ ]:
K = np.array([[260.8747863769531, 0, 321.9953308105469],
                           [0, 260.8747863769531, 179.68511962890625],
                           [0, 0, 1],
                            ], dtype=np.float32)

drive = 2
image_num = 2
depth_folder = f"/ssd1/jm_data/depth/ssl/monodepth2/jbnu_stereo/2025_05_08/2025_05_08_drive_{str(drive).zfill(4)}_sync/proj_depth/groundtruth_crm/image_0{image_num}"
save_folder = f"/ssd1/jm_data/depth/ssl/monodepth2/jbnu_stereo/2025_05_08/2025_05_08_drive_{str(drive).zfill(4)}_sync/proj_depth/groundtruth_consistence/image_0{image_num}"

process_consistency_folder(
    depth_folder=depth_folder,
    K=K,
    baseline=0.12,
    z_thresh=1.0,
    save_folder=save_folder,
    visualize=False  # True로 하면 하나씩 그림도 띄워줌
)


Processing depth maps: 100%|██████████| 1811/1811 [00:03<00:00, 454.12it/s]


In [94]:
import torch
import torch.nn.functional as F

def backproject_disparity(disparity, inv_K):
    """
    disparity: [B, 1, H, W]
    inv_K: [B, 3, 3]
    returns: [B, 3, H*W] cam_points
    """
    B, _, H, W = disparity.shape
    device = disparity.device

    # Disparity to depth (scale 없이)
    depth = 1.0 / (disparity + 1e-7)

    # 기존 backprojection 로직
    u = torch.arange(0, W, device=device)
    v = torch.arange(0, H, device=device)
    grid_u, grid_v = torch.meshgrid(u, v, indexing='xy')
    ones = torch.ones_like(grid_u)

    pix_coords = torch.stack([grid_u, grid_v, ones], dim=0).float()  # (3, H, W)
    pix_coords = pix_coords.reshape(3, -1).unsqueeze(0).repeat(B, 1, 1)  # [B, 3, H*W]

    cam_dirs = torch.bmm(inv_K, pix_coords)  # [B, 3, H*W]
    depth_flat = depth.view(B, 1, -1)  # [B, 1, H*W]
    cam_points = cam_dirs * depth_flat  # [B, 3, H*W]
    return cam_points, depth



def project_3d(cam_points, K, T, H, W):
    """
    cam_points: [B, 3, N]
    K: [B, 3, 3]
    T: [B, 4, 4]
    returns: grid for sampling [B, H, W, 2]
    """
    B, _, N = cam_points.shape
    ones = torch.ones((B, 1, N), device=cam_points.device)
    cam_points_homo = torch.cat([cam_points, ones], dim=1)  # [B, 4, N]

    P = torch.bmm(K, T[:, :3, :])  # [B, 3, 4]
    cam_points_proj = torch.bmm(P, cam_points_homo)  # [B, 3, N]
    cam_points_proj = cam_points_proj / (cam_points_proj[:, 2:3, :] + 1e-7)

    x = cam_points_proj[:, 0, :].view(B, H, W)
    y = cam_points_proj[:, 1, :].view(B, H, W)

    x_norm = 2 * (x / (W - 1)) - 1
    y_norm = 2 * (y / (H - 1)) - 1
    grid = torch.stack([x_norm, y_norm], dim=-1)  # [B, H, W, 2]
    return grid

def compute_3d_consistency_mask_from_disparity(disp_t, disp_tp1, K, T_t2tp1, threshold=1.0):
    """
    Inputs:
        disp_t:     [B, 1, H, W] disparity at time t
        disp_tp1:   [B, 1, H, W] disparity at time t+1
        K:          [B, 3, 3]
        T_t2tp1:    [B, 4, 4]
    """
    B, _, H, W = disp_t.shape
    device = disp_t.device
    inv_K = torch.inverse(K)

    # 1. disparity_t → depth → 3D
    cam_points_t, depth_t = backproject_disparity(disp_t, inv_K)  # [B, 3, H*W]

    # 2. transform and project
    grid = project_3d(cam_points_t, K, T_t2tp1, H, W)  # [B, H, W, 2]
    disp_tp1_sampled = F.grid_sample(disp_tp1, grid, mode='bilinear', padding_mode='border', align_corners=True)

    # 3. sampled disparity → depth → 3D
    cam_points_tp1, _ = backproject_disparity(disp_tp1_sampled, inv_K)  # [B, 3, H*W]
    ones = torch.ones((B, 1, H * W), device=device)
    cam_points_tp1_homo = torch.cat([cam_points_tp1, ones], dim=1)
    T_tp1_to_t = torch.inverse(T_t2tp1)
    cam_points_back = torch.bmm(T_tp1_to_t, cam_points_tp1_homo).view(B, 4, H, W)[:, :3]

    # 4. depth 비교 (Z difference)
    Z_t = depth_t.view(B, 1, H, W)
    Z_back = cam_points_back[:, 2:3]
    mask = torch.abs(Z_t - Z_back) < threshold  # [B, 1, H, W]
    return mask.float()



In [157]:
# D_t, D_tp1: predicted monocular depth [1, 1, H, W]
# K: camera intrinsic [1, 3, 3]
# T_t2tp1: predicted pose [1, 4, 4]

drive = 13
image_num = 3
depth_folder = f"/ssd1/jm_data/depth/ssl/monodepth2/jbnu_stereo/2025_05_08/2025_05_08_drive_{str(drive).zfill(4)}_sync/"
depth_path = os.path.join(depth_folder, f"proj_depth/groundtruth_crm/image_0{image_num}/")
extrinsic_path = os.path.join(depth_folder, f"extrinsic/image_0{image_num}/data/")
save_path = os.path.join(depth_folder, f"proj_depth/groundtruth_checked/image_0{image_num}/")
os.makedirs(save_path, exist_ok=True)

# ---------- intrinsic ----------
K_np = np.array([[260.8747863769531, 0, 321.9953308105469],
                 [0, 260.8747863769531, 179.68511962890625],
                 [0, 0, 1]], dtype=np.float32)
K = torch.from_numpy(K_np).unsqueeze(0).to("cuda")

file_list = sorted([f for f in os.listdir(depth_path) if f.endswith(".npy")])
n = len(file_list)

for i in tqdm(range(n - 1)):  # 마지막은 frame+1 없으므로 제외
    name0 = file_list[i]
    name1 = file_list[i + 1]

    # disparity 대신 depth로 불러왔다면 1/depth 취급
    D_t = np.load(os.path.join(depth_path, name0))
    D_tp1 = np.load(os.path.join(depth_path, name1))
    D_t_tensor = torch.from_numpy(D_t).unsqueeze(0).unsqueeze(0).to("cuda")
    D_tp1_tensor = torch.from_numpy(D_tp1).unsqueeze(0).unsqueeze(0).to("cuda")

    # extrinsic
    frame_idx0 = int(name0.replace(".npy", ""))
    frame_idx1 = int(name1.replace(".npy", ""))
    extrinsic_file = os.path.join(extrinsic_path, f"T_{frame_idx0:06d}_{frame_idx1:06d}.txt")
    if not os.path.exists(extrinsic_file):
        print("pass")
        continue  # skip if pose missing
    T = np.loadtxt(extrinsic_file)
    T_t2tp1 = torch.from_numpy(T).float().unsqueeze(0).to("cuda")

    # 3D 일관성 마스크
    mask = compute_3d_consistency_mask_from_disparity(D_t_tensor, D_tp1_tensor, K, T_t2tp1, threshold=5.0)

    # 마스크 적용한 depth 저장
    consistence_depth = D_t_tensor * mask
    consistence_depth_np = consistence_depth.squeeze().detach().cpu().numpy()
    save_name = os.path.join(save_path, f"{frame_idx0:010d}.npy")
    np.save(save_name, consistence_depth_np)

100%|██████████| 121/121 [00:00<00:00, 460.85it/s]
